# Phase 3 V1 document-sequence CV — Kaggle-ready, no Phase 1 retraining

This notebook loads the already-trained V1 checkpoint and preserves its original tokenizer and model definition. It freezes V1, compares document-level sequence models using five-fold document-grouped cross-validation, then refits the selected model on the labeled train and development documents and writes a Codabench-validated submission ZIP.

It does not repeat V1 language-model training. Test labels are never used. Cross-validation is used because the supplied development split has only five documents per domain; the notebook does not claim that a single small split guarantees a score.

## What this run changes

The previous best run reached mean fixed-13 development Macro-F1 0.3671 with a BiLSTM over frozen V1 vectors. The sentence-only model reached 0.2952, while its training loss fell sharply. That gap points to document context and generalization as the main issues.

This experiment compares (1) the existing final-token representation with a BiLSTM, (2) final-token plus mean/max token pooling with a BiLSTM, and (3) the pooled representation with a BiLSTM-CRF. Each uses the same document folds, class-weight rule, and fixed 13-label scoring. The selection score is aggregated out-of-fold mean domain Macro-F1.

For the final model, the selected architecture is trained on all labeled train and development documents for the median best epoch across folds. Attach the existing Kaggle input datasets containing bpe_en_25k.json, v1_model.pt, label_vocab.json, and the six CL/IT data files. No Hugging Face download or newly uploaded model is required.

In [ ]:
import os, json, math, random, time, copy, gc, hashlib, zipfile, itertools
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v1_sequence_cv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SENTENCE_TOKENS = 512
FEATURE_BATCH_SIZE = 4
CV_FOLDS = 5
CV_MAX_EPOCHS = 16
CV_PATIENCE = 4
FINAL_BATCH_DOCS = 2
CLASS_WEIGHT_POWER = 0.5
MAX_CLASS_WEIGHT = 3.0
CRF_AUX_CE_WEIGHT = 0.25
USE_MIXED_PRECISION = True

print("Device:", DEVICE)
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Find each supplied asset under Kaggle's mounted input datasets.
INPUT_ROOT = Path("/kaggle/input")
ASSET_OVERRIDES = {
    # Optional explicit paths if Kaggle has more than one file with the same name.
    # Example: "v1_model.pt": "/kaggle/input/my-model/v1_model.pt"
}
REQUIRED_FILES = [
    "bpe_en_25k.json", "v1_model.pt", "label_vocab.json", "CL_train.json", "CL_dev.json",
    "IT_train.json", "IT_dev.json", "CL_test.json", "IT_test.json"
]

def resolve_input_file(filename):
    override = ASSET_OVERRIDES.get(filename)
    if override:
        path = Path(override)
        if not path.is_file(): raise FileNotFoundError(f"Configured asset does not exist: {path}")
        return path
    if not INPUT_ROOT.exists():
        raise FileNotFoundError("/kaggle/input is missing. Run this notebook in Kaggle and attach the project input datasets.")
    hits = sorted(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    if not hits: raise FileNotFoundError(f"Required file {filename!r} was not found under {INPUT_ROOT}.")
    if len(hits) > 1:
        raise RuntimeError(f"Found multiple {filename!r} files. Set ASSET_OVERRIDES[{filename!r}] to the intended Kaggle path: {hits}")
    return hits[0]

ASSETS = {name: resolve_input_file(name) for name in REQUIRED_FILES}
for name, path in ASSETS.items(): print(f"{name}: {path}")

## Original V1 tokenizer loading

The following tokenizer loading and tokenizer implementation cells are copied verbatim from original V1 notebook cells 13 and 14. The original tokenizer uses the `itos`/`merges` payload and word-by-word byte-level BPE.

In [ ]:
#tokenizer from A1
import os
import json

BPE_FILENAME = "bpe_en_25k.json"
bpe_candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    if BPE_FILENAME in files:
        bpe_candidates.append(os.path.join(root, BPE_FILENAME))

if len(bpe_candidates) == 0:
    raise FileNotFoundError(
        f"{BPE_FILENAME} was not found inside /kaggle/input/. "
        "Make sure the BPE-25k dataset is attached."
    )

BPE_PATH = bpe_candidates[0]

print("Tokenizer found:", BPE_PATH)


# Load the vocabulary and learned BPE merges.
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]

# Make sure we have loaded the tokenizer we actually want.
assert bpe_vocab_size == 25_000
assert len(bpe_itos) == 25_000

print("Vocabulary size:", bpe_vocab_size)
print("Number of merge rules:", len(bpe_merges))


# Token -> ID mapping
bpe_stoi = {
    token: idx
    for idx, token in enumerate(bpe_itos)
}
merge_to_id = {}

for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge

    left_id = bpe_stoi[left_token]
    right_id = bpe_stoi[right_token]

    new_token_id = 261 + merge_index
    merge_to_id[(left_id, right_id)] = new_token_id

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

print("\nSpecial token IDs:")
print("PAD:", PAD_TOKEN_ID)
print("UNK:", UNK_TOKEN_ID)
print("BOS:", BOS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)
print("WORD BOUNDARY:", WORD_BOUNDARY_ID)

In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

In [ ]:
# Load and audit the Phase 3 label vocabulary and data schemas.
LABEL_VOCAB_PATH = ASSETS["label_vocab.json"]
with open(LABEL_VOCAB_PATH, "r", encoding="utf-8") as f: label_to_id = json.load(f)
label_to_id = {str(k): int(v) for k, v in label_to_id.items()}
assert len(label_to_id) == 13, f"Expected 13 labels, found {len(label_to_id)}"
assert sorted(label_to_id.values()) == list(range(13)), "label_vocab.json indices must be contiguous 0..12"
id_to_label = {i: label for label, i in label_to_id.items()}
label_set = set(label_to_id)
print("Official label mapping:", label_to_id)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def audit_labeled(path, split_name):
    records = load_json(path)
    if not isinstance(records, list): raise TypeError(f"{split_name}: expected a list of documents")
    ids=[]; sentence_count=0; empty_count=0; counts=Counter()
    for row_num, row in enumerate(records):
        if not isinstance(row, dict): raise TypeError(f"{split_name} row {row_num}: expected object")
        for key in ("id", "text", "labels"):
            if key not in row: raise ValueError(f"{split_name} row {row_num}: missing {key!r}")
        if not isinstance(row["text"], list) or not isinstance(row["labels"], list):
            raise TypeError(f"{split_name} row {row_num}: text and labels must be lists")
        if len(row["text"]) == 0: raise ValueError(f"{split_name} {row['id']}: empty document")
        if len(row["text"]) != len(row["labels"]):
            raise ValueError(f"{split_name} {row['id']}: {len(row['text'])} sentences but {len(row['labels'])} labels")
        ids.append(str(row["id"])); sentence_count += len(row["text"])
        empty_count += sum(not str(x).strip() for x in row["text"])
        unknown=set(row["labels"])-label_set
        if unknown: raise ValueError(f"{split_name} {row['id']}: unknown labels {sorted(unknown)}")
        counts.update(row["labels"])
    if len(ids) != len(set(ids)): raise ValueError(f"{split_name}: duplicate document IDs")
    print(f"{split_name}: {len(records)} documents, {sentence_count} sentences, {empty_count} empty sentences")
    if counts: print("  label counts:", dict(counts))
    return records, set(ids)

CL_TRAIN, cl_train_ids = audit_labeled(ASSETS["CL_train.json"], "CL train")
CL_DEV, cl_dev_ids = audit_labeled(ASSETS["CL_dev.json"], "CL dev")
IT_TRAIN, it_train_ids = audit_labeled(ASSETS["IT_train.json"], "IT train")
IT_DEV, it_dev_ids = audit_labeled(ASSETS["IT_dev.json"], "IT dev")
assert not (cl_train_ids & cl_dev_ids), "CL train/dev document leakage"
assert not (it_train_ids & it_dev_ids), "IT train/dev document leakage"

def document_fingerprints(records):
    return {hashlib.sha256(json.dumps([str(s) for s in row["text"]],ensure_ascii=False,
        separators=(",",":")).encode("utf-8")).hexdigest():str(row["id"]) for row in records}
named_splits={"CL_train":CL_TRAIN,"CL_dev":CL_DEV,"IT_train":IT_TRAIN,"IT_dev":IT_DEV}
fingerprints={name:document_fingerprints(rows) for name,rows in named_splits.items()}
for i,left in enumerate(named_splits):
    for right in list(named_splits)[i+1:]:
        overlap=set(fingerprints[left]) & set(fingerprints[right])
        if overlap: raise ValueError(f"Exact document-text overlap between {left} and {right}: {len(overlap)}")
print("No exact document-text duplicates across the four labeled train/dev splits.")

def load_test_docs(path, domain):
    data = load_json(path)
    if not isinstance(data, dict): raise TypeError(f"{domain} test: expected mapping document ID -> sentence list")
    docs=[]
    for doc_id, sentences in data.items():
        if not isinstance(sentences, list): raise TypeError(f"{domain} test {doc_id}: expected sentence list")
        docs.append((str(doc_id), [str(s) for s in sentences]))
    print(f"{domain} test: {len(docs)} documents, {sum(len(x[1]) for x in docs)} sentences")
    return docs

CL_TEST = load_test_docs(ASSETS["CL_test.json"], "CL")
IT_TEST = load_test_docs(ASSETS["IT_test.json"], "IT")
for domain,docs in (("CL",CL_TEST),("IT",IT_TEST)):
    empty=[doc_id for doc_id,sentences in docs if not sentences]
    if empty: raise ValueError(f"{domain} test contains empty documents: {empty[:5]}")


In [ ]:
# Match V1 pretraining convention: no BOS/EOS, and cap long sentences consistently.
def encode_sentence(sentence):
    ids = tokenizer.encode(str(sentence), add_bos=False, add_eos=False)
    if not ids: ids = [UNK_TOKEN_ID]
    truncated = len(ids) > MAX_SENTENCE_TOKENS
    if truncated: ids = ids[:MAX_SENTENCE_TOKENS]
    return ids, truncated

## Original V1 architecture

This is original V1 notebook cell 26 copied verbatim. It defines the pretrained V1 module names and parameter shapes. The following Phase 3 cells instantiate this exact class from the checkpoint configuration and load its saved weights strictly.

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
# Load the already-trained V1 checkpoint. This does not execute Phase 1 training.
CHECKPOINT_PATH = resolve_input_file("v1_model.pt")
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
    raise ValueError("Expected the supplied V1 checkpoint wrapper with model_state_dict")
model_config = checkpoint.get("config", {})
expected_config = {"vocab_size":25000,"n_embd":768,"n_layer":12,"n_head":12,
                   "n_kv_head":4,"ffn_multiplier":4,"context_length":1024}
for key, expected in expected_config.items():
    if key in model_config and int(model_config[key]) != expected:
        raise ValueError(f"Checkpoint config mismatch for {key}: {model_config[key]} vs {expected}")

backbone = V1LegalLM(
    vocab_size=int(model_config.get("vocab_size",25000)),
    n_embd=int(model_config.get("n_embd",768)),
    n_layer=int(model_config.get("n_layer",12)),
    n_head=int(model_config.get("n_head",12)),
    n_kv_head=int(model_config.get("n_kv_head",4)),
    max_seq_len=int(model_config.get("context_length",1024)),
    ffn_multiplier=int(model_config.get("ffn_multiplier",4)),
    dropout=float(model_config.get("dropout",0.0)))
load_result = backbone.load_state_dict(checkpoint["model_state_dict"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
backbone = backbone.to(DEVICE)
print("Strict V1 checkpoint load succeeded.")
print("Checkpoint:", CHECKPOINT_PATH)
print("Best Phase 1 validation loss:", checkpoint.get("best_val_loss"))
print("Backbone parameters:", f"{sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# Reuse the V1 token embedding, Transformer blocks, and final RMSNorm.
# This is the same hidden-state path used inside V1LegalLM.forward, omitting only
# the 25,000-way language-model projection. Verify that equivalence below.
def encode_v1_hidden(v1_model, input_ids):
    x = v1_model.token_embedding(input_ids)
    for block in v1_model.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)
    return v1_model.final_norm(x)

probe_ids=torch.tensor([[4,5,6,7,8]],dtype=torch.long,device=DEVICE)
backbone.eval()
with torch.no_grad():
    full_logits,_=backbone(probe_ids,use_cache=False)
    hidden=encode_v1_hidden(backbone,probe_ids)
    projected_logits=backbone.lm_head(hidden)
assert torch.allclose(full_logits,projected_logits,atol=1e-5,rtol=1e-5)
print("V1 hidden extraction matches original forward path; max difference:",
      float((full_logits-projected_logits).abs().max()))

class V1RhetoricalRolePredictor(nn.Module):
    def __init__(self, pretrained_backbone, n_labels=13, dropout=0.10):
        super().__init__()
        self.backbone=pretrained_backbone
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(self.backbone.n_embd,n_labels)
    def forward(self,input_ids,lengths):
        hidden=encode_v1_hidden(self.backbone,input_ids)
        rows=torch.arange(input_ids.shape[0],device=input_ids.device)
        sentence_state=hidden[rows,lengths.to(input_ids.device)-1]
        return self.classifier(self.dropout(sentence_state))

rr_model=V1RhetoricalRolePredictor(backbone,len(label_to_id),HEAD_DROPOUT).to(DEVICE)
print("RR output classes:",rr_model.classifier.out_features)

In [ ]:
# Fixed-label metric: all 13 official roles count, including unsupported labels.
N_LABELS = len(label_to_id)
def metrics_from_ids(gold, pred):
    if len(gold) != len(pred) or not gold:
        raise ValueError("Metric inputs must be non-empty and have equal lengths")
    cm = np.zeros((N_LABELS, N_LABELS), dtype=np.int64)
    for y, p in zip(gold, pred): cm[int(y), int(p)] += 1
    tp = np.diag(cm).astype(float)
    support = cm.sum(axis=1).astype(float)
    predicted = cm.sum(axis=0).astype(float)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros_like(tp), where=(precision+recall) > 0)
    active = support > 0
    per_class = {id_to_label[i]: {
        "precision": float(precision[i]), "recall": float(recall[i]),
        "f1": float(f1[i]), "support": int(support[i]),
        "predicted_count": int(predicted[i])} for i in range(N_LABELS)}
    return {"macro_precision_all_13": float(precision.mean()),
            "macro_recall_all_13": float(recall.mean()),
            "macro_f1": float(f1.mean()),
            "macro_f1_gold_supported_diagnostic": float(f1[active].mean()) if active.any() else 0.0,
            "gold_supported_label_count": int(active.sum()),
            "per_class": per_class, "confusion_matrix": cm.tolist()}

## Frozen V1 feature extraction

The next cell loads no new language model. It extracts three views from the trained V1 hidden states: the final non-padding token, the mean across non-padding tokens, and the maximum across non-padding tokens. This gives the sequence model access to sentence-ending context and information distributed across the sentence.

In [ ]:
backbone = backbone.to(DEVICE).eval()
for parameter in backbone.parameters(): parameter.requires_grad_(False)

def make_document(record, domain, split, test=False):
    if test:
        doc_id, sentences = record
        labels = None
    else:
        doc_id, sentences = str(record["id"]), record["text"]
        labels = [label_to_id[x] for x in record["labels"]]
        if len(sentences) != len(labels):
            raise ValueError(f"{domain}/{split}/{doc_id}: sentence/label alignment mismatch")
    if not sentences: raise ValueError(f"{domain}/{split}/{doc_id}: empty document")
    return {"key": f"{domain}::{doc_id}", "id": str(doc_id), "domain": domain,
            "split": split, "sentences": [str(x) for x in sentences], "labels": labels}

DOCUMENTS = []
for domain, split_name, rows in [
    ("CL", "train", CL_TRAIN), ("CL", "dev", CL_DEV),
    ("IT", "train", IT_TRAIN), ("IT", "dev", IT_DEV)]:
    DOCUMENTS.extend(make_document(row, domain, split_name) for row in rows)
for domain, rows in [("CL", CL_TEST), ("IT", IT_TEST)]:
    DOCUMENTS.extend(make_document(row, domain, "test", test=True) for row in rows)

all_keys = [row["key"] for row in DOCUMENTS]
if len(all_keys) != len(set(all_keys)): raise ValueError("Duplicate domain/document key")
test_ids = [row["id"] for row in DOCUMENTS if row["split"] == "test"]
if len(test_ids) != len(set(test_ids)):
    raise ValueError("Test document IDs must be unique across CL and IT for flat Codabench JSON")

def extract_document_features(documents):
    examples = []
    total_truncated = 0
    for doc in documents:
        for sentence_index, sentence in enumerate(doc["sentences"]):
            ids, truncated = encode_sentence(sentence)
            total_truncated += int(truncated)
            examples.append((doc["key"], sentence_index, ids))
    grouped = defaultdict(list)
    for start in range(0, len(examples), FEATURE_BATCH_SIZE):
        batch = examples[start:start+FEATURE_BATCH_SIZE]
        seqs = [torch.tensor(row[2], dtype=torch.long) for row in batch]
        lengths = torch.tensor([len(x) for x in seqs], dtype=torch.long, device=DEVICE)
        input_ids = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN_ID).to(DEVICE)
        with torch.inference_mode():
            if DEVICE.type == "cuda" and USE_MIXED_PRECISION:
                dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
                with torch.autocast(device_type="cuda", dtype=dtype):
                    hidden = encode_v1_hidden(backbone, input_ids)
            else:
                hidden = encode_v1_hidden(backbone, input_ids)
            hidden = hidden.float()
            rows = torch.arange(input_ids.shape[0], device=DEVICE)
            last = hidden[rows, lengths-1]
            mask = torch.arange(hidden.shape[1], device=DEVICE)[None, :] < lengths[:, None]
            mean = (hidden * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(-1)
            maximum = hidden.masked_fill(~mask.unsqueeze(-1), -torch.inf).amax(dim=1)
            multi = torch.cat([last, mean, maximum], dim=-1)
        for j, (key, sentence_index, _) in enumerate(batch):
            grouped[key].append((sentence_index, last[j].cpu(), multi[j].cpu()))

    result = {}
    for doc in documents:
        rows = sorted(grouped[doc["key"]], key=lambda x: x[0])
        if len(rows) != len(doc["sentences"]):
            raise RuntimeError(f"{doc['key']}: feature count does not match sentence count")
        item = dict(doc)
        item["features_last"] = torch.stack([x[1] for x in rows]).contiguous()
        item["features_multi"] = torch.stack([x[2] for x in rows]).contiguous()
        if doc["labels"] is not None:
            item["labels_tensor"] = torch.tensor(doc["labels"], dtype=torch.long)
        result[doc["key"]] = item
    print(f"Frozen V1 features: {len(result)} documents, {len(examples)} sentences, "
          f"{total_truncated} sentences truncated at {MAX_SENTENCE_TOKENS} BPE tokens")
    return result

DOC_FEATURES = extract_document_features(DOCUMENTS)
del rr_model, backbone, checkpoint
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
def make_balanced_domain_folds(domain_docs, n_folds=5, seed=42, restarts=12):
    # Greedy multilabel document assignment; no sentence or document is split across folds.
    if len(domain_docs) < n_folds: raise ValueError("Each domain needs at least one document per fold")
    vectors = np.stack([
        np.bincount(row["labels"], minlength=N_LABELS).astype(float)
        for row in domain_docs])
    totals = vectors.sum(axis=0)
    targets = totals / n_folds
    doc_freq = (vectors > 0).sum(axis=0)
    target_size = len(domain_docs) / n_folds
    best_assignment, best_cost = None, float("inf")
    for attempt in range(restarts):
        rng = np.random.default_rng(seed + attempt)
        tie = rng.random(len(domain_docs))
        order = sorted(range(len(domain_docs)), key=lambda i: (
            int(doc_freq[vectors[i] > 0].min()) if np.any(vectors[i] > 0) else 10**9,
            -int(vectors[i].sum()), tie[i]))
        fold_counts = np.zeros((n_folds, N_LABELS), dtype=float)
        fold_sizes = np.zeros(n_folds, dtype=int)
        assignment = np.full(len(domain_docs), -1, dtype=int)
        for i in order:
            eligible = np.flatnonzero(fold_sizes == fold_sizes.min())
            scores = []
            for f in eligible:
                after = fold_counts[f] + vectors[i]
                normalized = (after - targets) / (targets + 1.0)
                size_error = ((fold_sizes[f] + 1 - target_size) / (target_size + 1.0))**2
                scores.append(float(np.square(normalized).sum() + 0.1*size_error + rng.random()*1e-9))
            f = int(eligible[int(np.argmin(scores))])
            assignment[i] = f
            fold_counts[f] += vectors[i]
            fold_sizes[f] += 1
        # Refine with document swaps. This keeps fold sizes fixed while reducing
        # count imbalance caused by unusually long legal opinions.
        def fold_cost(counts):
            return float(np.square((counts - targets) / (targets + 1.0)).sum())
        current_cost = fold_cost(fold_counts)
        for _ in range(100):
            best_swap, next_cost = None, current_cost
            for i in range(len(domain_docs)):
                for j in range(i+1, len(domain_docs)):
                    a, b = int(assignment[i]), int(assignment[j])
                    if a == b: continue
                    after_a = fold_counts[a] - vectors[i] + vectors[j]
                    after_b = fold_counts[b] - vectors[j] + vectors[i]
                    old_cost = np.square((fold_counts[a]-targets)/(targets+1.0)).sum()
                    old_cost += np.square((fold_counts[b]-targets)/(targets+1.0)).sum()
                    new_cost = np.square((after_a-targets)/(targets+1.0)).sum()
                    new_cost += np.square((after_b-targets)/(targets+1.0)).sum()
                    candidate_cost = current_cost - old_cost + new_cost
                    if candidate_cost < next_cost - 1e-10:
                        next_cost = float(candidate_cost)
                        best_swap = (i, j, a, b, after_a, after_b)
            if best_swap is None: break
            i, j, a, b, after_a, after_b = best_swap
            assignment[i], assignment[j] = b, a
            fold_counts[a], fold_counts[b] = after_a, after_b
            current_cost = next_cost
        cost = current_cost
        if cost < best_cost:
            best_cost, best_assignment = cost, assignment.copy()
    return [[domain_docs[i] for i in range(len(domain_docs)) if best_assignment[i] == f]
            for f in range(n_folds)]

CV_DOCS = {
    domain: [DOC_FEATURES[f"{domain}::{str(row['id'])}"] for row in rows]
    for domain, rows in [("CL", CL_TRAIN), ("IT", IT_TRAIN)]}
FOLDS = {d: make_balanced_domain_folds(rows, CV_FOLDS, SEED + j*100)
         for j, (d, rows) in enumerate(CV_DOCS.items())}
for domain in ("CL", "IT"):
    sizes = [len(fold) for fold in FOLDS[domain]]
    assert sum(sizes) == len(CV_DOCS[domain]) and len({x["key"] for f in FOLDS[domain] for x in f}) == len(CV_DOCS[domain])
    print(domain, "document folds:", sizes)
assert all(len(FOLDS["CL"][i]) == len(FOLDS["IT"][i]) for i in range(CV_FOLDS)),        "Paired domain folds must have equal document counts"

def class_weights_from_docs(documents):
    counts = {}
    for domain in ("CL", "IT"):
        domain_docs = [row for row in documents if row["domain"] == domain]
        if not domain_docs: raise ValueError(f"No training documents for {domain}")
        counts[domain] = sum(
            (np.bincount(row["labels"], minlength=N_LABELS).astype(float) for row in domain_docs),
            start=np.zeros(N_LABELS, dtype=float))
    proportions = []
    for domain in ("CL", "IT"):
        total = counts[domain].sum()
        if total <= 0: raise ValueError(f"No labeled sentences for {domain}")
        proportions.append(counts[domain] / total)
    reference = np.mean(proportions, axis=0)
    present = reference > 0
    weights = np.zeros(N_LABELS, dtype=np.float32)
    weights[present] = np.minimum(
        (reference[present].mean() / reference[present]) ** CLASS_WEIGHT_POWER,
        MAX_CLASS_WEIGHT)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def collate_feature_docs(rows, feature_mode):
    feature_key = "features_last" if feature_mode == "last" else "features_multi"
    features = pad_sequence([row[feature_key] for row in rows], batch_first=True, padding_value=0.0)
    lengths = torch.tensor([len(row["labels"]) for row in rows], dtype=torch.long)
    labels = pad_sequence([row["labels_tensor"] for row in rows], batch_first=True, padding_value=-100)
    return features.to(DEVICE, non_blocking=True), lengths, labels.to(DEVICE, non_blocking=True)

class LinearChainCRF(nn.Module):
    # transitions[i,j] is the score for moving from previous label i to next label j.
    def __init__(self, n_labels):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(n_labels, n_labels))
        self.start_transitions = nn.Parameter(torch.zeros(n_labels))
        self.end_transitions = nn.Parameter(torch.zeros(n_labels))

    def neg_log_likelihood(self, emissions, tags):
        if emissions.ndim != 2 or tags.ndim != 1 or emissions.shape[0] != tags.shape[0]:
            raise ValueError("CRF expects emissions [time, labels] and tags [time]")
        if emissions.shape[0] == 0: raise ValueError("CRF cannot decode an empty document")
        emissions = emissions.float()
        tags = tags.long()
        alpha = self.start_transitions.float() + emissions[0]
        for t in range(1, emissions.shape[0]):
            alpha = torch.logsumexp(alpha[:, None] + self.transitions.float(), dim=0) + emissions[t]
        log_partition = torch.logsumexp(alpha + self.end_transitions.float(), dim=0)
        gold = self.start_transitions.float()[tags[0]] + emissions[0, tags[0]]
        for t in range(1, emissions.shape[0]):
            gold = gold + self.transitions.float()[tags[t-1], tags[t]] + emissions[t, tags[t]]
        gold = gold + self.end_transitions.float()[tags[-1]]
        return log_partition - gold

    @torch.inference_mode()
    def decode(self, emissions):
        if emissions.ndim != 2 or emissions.shape[0] == 0:
            raise ValueError("CRF decode expects non-empty emissions [time, labels]")
        emissions = emissions.float()
        score = self.start_transitions.float() + emissions[0]
        backpointers = []
        for t in range(1, emissions.shape[0]):
            candidates = score[:, None] + self.transitions.float()
            best_score, best_prev = candidates.max(dim=0)
            score = best_score + emissions[t]
            backpointers.append(best_prev)
        best_last = int((score + self.end_transitions.float()).argmax().item())
        path = [best_last]
        for back in reversed(backpointers):
            best_last = int(back[best_last].item())
            path.append(best_last)
        path.reverse()
        return path

# Tiny exhaustive check: compare the dynamic program with every possible path.
_crf_check = LinearChainCRF(3)
_crf_emissions = torch.randn(3, 3)
_crf_tags = torch.tensor([2, 0, 1], dtype=torch.long)
_all_paths = list(itertools.product(range(3), repeat=3))
_path_scores = []
for _path in _all_paths:
    _score = _crf_check.start_transitions[_path[0]] + _crf_emissions[0, _path[0]]
    for _t in range(1, 3):
        _score = _score + _crf_check.transitions[_path[_t-1], _path[_t]] + _crf_emissions[_t, _path[_t]]
    _path_scores.append(_score + _crf_check.end_transitions[_path[-1]])
_brute_log_partition = torch.logsumexp(torch.stack(_path_scores), dim=0)
_gold_score = _path_scores[_all_paths.index(tuple(_crf_tags.tolist()))]
_expected_nll = _brute_log_partition - _gold_score
_actual_nll = _crf_check.neg_log_likelihood(_crf_emissions, _crf_tags)
assert torch.allclose(_actual_nll, _expected_nll, atol=1e-6, rtol=1e-6)
assert len(_crf_check.decode(_crf_emissions)) == 3
print("CRF forward/backward dynamic-programming self-check passed.")

class SequenceTagger(nn.Module):
    def __init__(self, feature_mode, use_crf, n_labels=N_LABELS):
        super().__init__()
        self.feature_mode = feature_mode
        self.use_crf = use_crf
        in_dim = 768 if feature_mode == "last" else 2304
        if feature_mode == "last":
            self.project = nn.Identity()
        else:
            self.project = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, 768),
                                         nn.GELU(), nn.Dropout(0.20))
        self.context = nn.LSTM(768, 256, num_layers=2, batch_first=True,
                               bidirectional=True, dropout=0.25)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(512, n_labels)
        self.crf = LinearChainCRF(n_labels) if use_crf else None

    def forward(self, features, lengths):
        x = self.project(features)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.context(packed)
        sequence, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=features.shape[1])
        return self.classifier(self.dropout(sequence))

CANDIDATES = {
    "last_bilstm_ce": {"feature_mode": "last", "use_crf": False},
    "multipool_bilstm_ce": {"feature_mode": "multi", "use_crf": False},
    "multipool_bilstm_crf": {"feature_mode": "multi", "use_crf": True},
}

## Document-grouped cross-validation

All folds hold out whole documents, with each domain balanced separately. The fold allocator also tries to distribute per-label sentence counts, while keeping every document intact. The model is selected using aggregated out-of-fold predictions. The CV score is an estimate from the supplied training documents; it is not a score on the hidden test labels.

In [ ]:
def predict_document_batch(model, rows, feature_mode, use_crf):
    features, lengths, _ = collate_feature_docs(rows, feature_mode)
    model.eval()
    with torch.inference_mode():
        logits = model(features, lengths)
    predictions = []
    for i, row in enumerate(rows):
        n = len(row["labels"])
        if use_crf:
            ids = model.crf.decode(logits[i, :n])
        else:
            ids = logits[i, :n].argmax(dim=-1).cpu().tolist()
        if len(ids) != n: raise RuntimeError(f"{row['key']}: decoded sentence count mismatch")
        predictions.append([int(x) for x in ids])
    return predictions

@torch.inference_mode()
def evaluate_documents(model, docs_by_domain, feature_mode, use_crf, batch_docs=4):
    result = {}
    all_predictions = {}
    for domain in ("CL", "IT"):
        gold, pred = [], []
        rows = docs_by_domain[domain]
        for start in range(0, len(rows), batch_docs):
            batch = rows[start:start+batch_docs]
            outputs = predict_document_batch(model, batch, feature_mode, use_crf)
            for row, ids in zip(batch, outputs):
                if len(ids) != len(row["labels"]): raise RuntimeError("Prediction/gold alignment error")
                gold.extend(row["labels"]); pred.extend(ids)
                all_predictions[row["key"]] = ids
        result[domain] = metrics_from_ids(gold, pred)
    return result, all_predictions

def paired_batches(docs_by_domain, feature_mode, seed):
    cl = docs_by_domain["CL"]; it = docs_by_domain["IT"]
    if len(cl) != len(it):
        raise ValueError(f"Equal-domain document updates required; got CL={len(cl)}, IT={len(it)}")
    rng = np.random.default_rng(seed)
    cl_order = rng.permutation(len(cl)); it_order = rng.permutation(len(it))
    for a, b in zip(cl_order, it_order):
        rows = [cl[int(a)], it[int(b)]]
        features, lengths, labels = collate_feature_docs(rows, feature_mode)
        yield rows, features, lengths, labels

def loss_for_pair(model, logits, labels, lengths, weights, use_crf):
    domain_losses = []
    for i in range(labels.shape[0]):
        n = int(lengths[i])
        y = labels[i, :n]
        if use_crf:
            seq_nll = model.crf.neg_log_likelihood(logits[i, :n], y) / max(n, 1)
            aux_tokens = F.cross_entropy(logits[i, :n].float(), y, weight=weights, reduction="none")
            aux = aux_tokens.sum() / weights[y].sum().clamp_min(1e-12)
            domain_losses.append(seq_nll + CRF_AUX_CE_WEIGHT * aux)
        else:
            token_losses = F.cross_entropy(logits[i, :n].float(), y, weight=weights, reduction="none")
            domain_losses.append(token_losses.sum() / weights[y].sum().clamp_min(1e-12))
    return torch.stack(domain_losses).mean()

def train_one_fold(candidate_name, candidate, train_by_domain, valid_by_domain, fold_id):
    seed = SEED + list(CANDIDATES).index(candidate_name)*1000 + fold_id*31
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model = SequenceTagger(**candidate).to(DEVICE)
    weights = class_weights_from_docs(train_by_domain["CL"] + train_by_domain["IT"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-3)
    best_score, best_epoch, best_state, best_metrics, best_predictions = -1.0, None, None, None, None
    stale, history = 0, []
    for epoch in range(1, CV_MAX_EPOCHS+1):
        model.train()
        total_loss, steps = 0.0, 0
        for rows, features, lengths, labels in paired_batches(
                train_by_domain, candidate["feature_mode"], seed + epoch):
            optimizer.zero_grad(set_to_none=True)
            logits = model(features, lengths)
            loss = loss_for_pair(model, logits, labels, lengths, weights, candidate["use_crf"])
            if not torch.isfinite(loss): raise FloatingPointError(f"{candidate_name}: non-finite loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.detach()); steps += 1
        metrics, predictions = evaluate_documents(
            model, valid_by_domain, candidate["feature_mode"], candidate["use_crf"])
        score = (metrics["CL"]["macro_f1"] + metrics["IT"]["macro_f1"]) / 2
        history.append({"epoch": epoch, "mean_pair_loss": total_loss/max(steps, 1),
                        "selection_mean_domain_macro_f1": score,
                        "CL_macro_f1": metrics["CL"]["macro_f1"],
                        "IT_macro_f1": metrics["IT"]["macro_f1"]})
        print(f"{candidate_name} | fold {fold_id+1}/{CV_FOLDS} | epoch {epoch} | "
              f"loss {history[-1]['mean_pair_loss']:.4f} | CL {metrics['CL']['macro_f1']:.4f} | "
              f"IT {metrics['IT']['macro_f1']:.4f} | mean {score:.4f}")
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = metrics
            best_predictions = predictions
            stale = 0
        else:
            stale += 1
            if stale >= CV_PATIENCE:
                print("Early stopping for this fold.")
                break
    return {"best_score": best_score, "best_epoch": best_epoch, "state": best_state,
            "metrics": best_metrics, "predictions": best_predictions, "history": history}

CV_RESULTS = {}
for candidate_name, candidate in CANDIDATES.items():
    print("\n===== Cross-validation candidate:", candidate_name, "=====")
    candidate_oof = {"CL": {}, "IT": {}}
    fold_results = []
    for fold_id in range(CV_FOLDS):
        valid_by_domain = {d: FOLDS[d][fold_id] for d in ("CL", "IT")}
        train_by_domain = {
            d: [row for i, fold in enumerate(FOLDS[d]) if i != fold_id for row in fold]
            for d in ("CL", "IT")}
        fold = train_one_fold(candidate_name, candidate, train_by_domain, valid_by_domain, fold_id)
        fold_results.append({k: fold[k] for k in ("best_score", "best_epoch", "metrics", "history")})
        for domain in ("CL", "IT"):
            for row in valid_by_domain[domain]:
                candidate_oof[domain][row["key"]] = fold["predictions"][row["key"]]
    oof_metrics = {}
    for domain in ("CL", "IT"):
        rows = CV_DOCS[domain]
        if set(candidate_oof[domain]) != {row["key"] for row in rows}:
            raise RuntimeError(f"{candidate_name}/{domain}: out-of-fold coverage is incomplete")
        gold = [y for row in rows for y in row["labels"]]
        pred = [y for row in rows for y in candidate_oof[domain][row["key"]]]
        oof_metrics[domain] = metrics_from_ids(gold, pred)
    oof_score = (oof_metrics["CL"]["macro_f1"] + oof_metrics["IT"]["macro_f1"]) / 2
    CV_RESULTS[candidate_name] = {
        "candidate": candidate, "oof_score": oof_score, "oof_metrics": oof_metrics,
        "fold_results": fold_results, "best_epochs": [x["best_epoch"] for x in fold_results],
        "oof_predictions": candidate_oof}
    print(f"{candidate_name} aggregated OOF mean domain Macro-F1: {oof_score:.4f}")

if not CV_RESULTS: raise RuntimeError("No candidate completed cross-validation")
SELECTED_NAME = max(CV_RESULTS, key=lambda name: CV_RESULTS[name]["oof_score"])
SELECTED_CANDIDATE = CANDIDATES[SELECTED_NAME]
SELECTED_EPOCHS = max(1, int(np.median(CV_RESULTS[SELECTED_NAME]["best_epochs"])))
print("\nSelected by aggregated OOF mean fixed-13 Macro-F1:", SELECTED_NAME)
print("Median fold best epoch for final refit:", SELECTED_EPOCHS)

cv_summary = {
    name: {"oof_score": value["oof_score"], "oof_metrics": value["oof_metrics"],
           "best_epochs": value["best_epochs"], "folds": value["fold_results"]}
    for name, value in CV_RESULTS.items()}
with open(OUTPUT_DIR/"cross_validation_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"folds": CV_FOLDS, "selection_rule": "aggregated out-of-fold mean of CL and IT fixed-13 Macro-F1",
               "selected_model": SELECTED_NAME, "selected_median_epoch": SELECTED_EPOCHS,
               "candidates": cv_summary}, f, ensure_ascii=False, indent=2)
oof_export = {
    domain: [{"id": DOC_FEATURES[key]["id"], "gold": [id_to_label[x] for x in DOC_FEATURES[key]["labels"]],
              "predictions": [id_to_label[x] for x in CV_RESULTS[SELECTED_NAME]["oof_predictions"][domain][key]]}
             for key in CV_RESULTS[SELECTED_NAME]["oof_predictions"][domain]]
    for domain in ("CL", "IT")}
with open(OUTPUT_DIR/"cross_validation_predictions.json", "w", encoding="utf-8") as f:
    json.dump(oof_export, f, ensure_ascii=False, indent=2)

## Final refit and Codabench export

After CV selects a candidate and a median stopping epoch, the notebook refits on all labeled train and development documents. It then predicts each test sentence once and writes diagnostics plus a flat document-ID JSON. The final cell reopens the ZIP and checks IDs, sentence counts, label vocabulary, archive member name, and CRC before reporting PASS.

In [ ]:
ALL_LABELED = {
    domain: [DOC_FEATURES[f"{domain}::{str(row['id'])}"]
             for rows in (train_rows, dev_rows) for row in rows]
    for domain, train_rows, dev_rows in [
        ("CL", CL_TRAIN, CL_DEV), ("IT", IT_TRAIN, IT_DEV)]}
FINAL_WEIGHTS = class_weights_from_docs(ALL_LABELED["CL"] + ALL_LABELED["IT"])
random.seed(SEED + 9999); np.random.seed(SEED + 9999); torch.manual_seed(SEED + 9999)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED + 9999)
FINAL_MODEL = SequenceTagger(**SELECTED_CANDIDATE).to(DEVICE)
FINAL_OPTIMIZER = torch.optim.AdamW(FINAL_MODEL.parameters(), lr=8e-4, weight_decay=1e-3)
FINAL_HISTORY = []
for epoch in range(1, SELECTED_EPOCHS + 1):
    FINAL_MODEL.train()
    total_loss, steps = 0.0, 0
    for rows, features, lengths, labels in paired_batches(
            ALL_LABELED, SELECTED_CANDIDATE["feature_mode"], SEED + 9999 + epoch):
        FINAL_OPTIMIZER.zero_grad(set_to_none=True)
        logits = FINAL_MODEL(features, lengths)
        loss = loss_for_pair(FINAL_MODEL, logits, labels, lengths, FINAL_WEIGHTS,
                             SELECTED_CANDIDATE["use_crf"])
        if not torch.isfinite(loss): raise FloatingPointError("Final refit produced non-finite loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(FINAL_MODEL.parameters(), 1.0)
        FINAL_OPTIMIZER.step()
        total_loss += float(loss.detach()); steps += 1
    row = {"epoch": epoch, "mean_pair_loss": total_loss/max(steps, 1)}
    FINAL_HISTORY.append(row)
    print(f"Final refit epoch {epoch}/{SELECTED_EPOCHS} | loss {row['mean_pair_loss']:.4f}")

CHECKPOINT_OUT = OUTPUT_DIR/"best_phase3_v1_cv_sequence.pt"
torch.save({"model_state_dict": FINAL_MODEL.state_dict(),
            "candidate": SELECTED_CANDIDATE, "candidate_name": SELECTED_NAME,
            "label_to_id": label_to_id, "v1_config": model_config,
            "class_weights": FINAL_WEIGHTS.detach().cpu().tolist(),
            "final_epochs": SELECTED_EPOCHS,
            "oof_mean_domain_macro_f1": CV_RESULTS[SELECTED_NAME]["oof_score"]}, CHECKPOINT_OUT)
with open(OUTPUT_DIR/"final_training_history.json", "w", encoding="utf-8") as f:
    json.dump(FINAL_HISTORY, f, ensure_ascii=False, indent=2)

@torch.inference_mode()
def predict_test_documents(domain):
    FINAL_MODEL.eval()
    rows = [DOC_FEATURES[f"{domain}::{doc_id}"] for doc_id, _ in
            (CL_TEST if domain == "CL" else IT_TEST)]
    output = []
    for start in range(0, len(rows), 4):
        batch = rows[start:start+4]
        # Test docs have no labels; make a label-free collator for inference.
        feature_key = "features_last" if SELECTED_CANDIDATE["feature_mode"] == "last" else "features_multi"
        features = pad_sequence([r[feature_key] for r in batch], batch_first=True, padding_value=0.0).to(DEVICE)
        lengths = torch.tensor([len(r["sentences"]) for r in batch], dtype=torch.long)
        logits = FINAL_MODEL(features, lengths)
        for i, row in enumerate(batch):
            n = len(row["sentences"])
            ids = FINAL_MODEL.crf.decode(logits[i, :n]) if SELECTED_CANDIDATE["use_crf"]                   else logits[i, :n].argmax(dim=-1).cpu().tolist()
            labels = [id_to_label[int(x)] for x in ids]
            if len(labels) != n or not set(labels) <= label_set:
                raise ValueError(f"{row['key']}: invalid output labels or length")
            output.append({"id": row["id"], "predictions": labels})
    return output

CL_PREDICTIONS = predict_test_documents("CL")
IT_PREDICTIONS = predict_test_documents("IT")
for domain, rows, expected in [
    ("CL", CL_PREDICTIONS, CL_TEST), ("IT", IT_PREDICTIONS, IT_TEST)]:
    if [x["id"] for x in rows] != [x[0] for x in expected]:
        raise ValueError(f"{domain}: output IDs or order differ from supplied test data")
    with open(OUTPUT_DIR/f"{domain}_test_predictions.json", "w", encoding="utf-8") as f:
        json.dump({"domain": domain, "documents": rows}, f, ensure_ascii=False, indent=2)

domain_diagnostics = {"CL": CL_PREDICTIONS, "IT": IT_PREDICTIONS}
with open(OUTPUT_DIR/"phase3_rr_predictions_by_domain.json", "w", encoding="utf-8") as f:
    json.dump(domain_diagnostics, f, ensure_ascii=False, indent=2)

expected_lengths = {}
for domain_docs in (CL_TEST, IT_TEST):
    for doc_id, sentences in domain_docs:
        key = str(doc_id)
        if key in expected_lengths: raise ValueError(f"Duplicate test document ID: {key}")
        expected_lengths[key] = len(sentences)
submission = {}
for rows in (CL_PREDICTIONS, IT_PREDICTIONS):
    for row in rows:
        key = str(row["id"]); labels = list(row["predictions"])
        if key in submission: raise ValueError(f"Duplicate prediction document ID: {key}")
        if key not in expected_lengths: raise ValueError(f"Unexpected prediction ID: {key}")
        if len(labels) != expected_lengths[key]: raise ValueError(f"{key}: prediction length mismatch")
        if any(not isinstance(x, str) for x in labels) or not set(labels) <= label_set:
            raise ValueError(f"{key}: invalid label value")
        submission[key] = labels
if set(submission) != set(expected_lengths):
    raise ValueError("Submission document IDs do not exactly match the test data")

PREDICTION_PATH = OUTPUT_DIR/"prediction.json"
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

def validate_submission(payload, source):
    if not isinstance(payload, dict): raise TypeError(f"{source}: expected document-ID object")
    if set(payload) != set(expected_lengths):
        if set(payload) == {"CL", "IT"}:
            raise ValueError(f"{source}: found domain-wrapped diagnostics; expected flat document IDs")
        raise ValueError(f"{source}: document IDs do not match the supplied test data")
    for doc_id, labels in payload.items():
        if not isinstance(labels, list): raise TypeError(f"{source}/{doc_id}: expected a label list")
        if len(labels) != expected_lengths[doc_id]:
            raise ValueError(f"{source}/{doc_id}: expected {expected_lengths[doc_id]} labels, got {len(labels)}")
        if any(not isinstance(x, str) for x in labels) or not set(labels) <= label_set:
            raise ValueError(f"{source}/{doc_id}: invalid label outside official vocabulary")

with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
    disk_payload = json.load(f)
validate_submission(disk_payload, "prediction.json")
if disk_payload != submission: raise ValueError("Serialized submission differs from validated output")

ZIP_PATH = OUTPUT_DIR/"submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(PREDICTION_PATH, arcname="prediction.json")
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    if archive.namelist() != ["prediction.json"]: raise ValueError(f"Unexpected ZIP members: {archive.namelist()}")
    if archive.testzip() is not None: raise ValueError("ZIP CRC validation failed")
    packaged = json.loads(archive.read("prediction.json").decode("utf-8"))
validate_submission(packaged, "submission.zip/prediction.json")
if packaged != disk_payload: raise ValueError("ZIP payload differs from prediction.json")

comparison = {
    "selected_model": SELECTED_NAME,
    "selected_oof_mean_domain_macro_f1": CV_RESULTS[SELECTED_NAME]["oof_score"],
    "selected_final_epochs": SELECTED_EPOCHS,
    "candidate_oof_scores": {name: value["oof_score"] for name, value in CV_RESULTS.items()},
    "training_data": "all supplied CL/IT train and development documents",
    "phase1_v1_retrained": False,
    "prediction_file": str(PREDICTION_PATH),
    "submission_zip": str(ZIP_PATH),
    "submission_documents": len(submission),
    "selection_rule": "highest aggregated out-of-fold mean of CL and IT fixed-13 Macro-F1"}
with open(OUTPUT_DIR/"model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print("Selected model:", SELECTED_NAME)
print("OOF mean domain fixed-13 Macro-F1:", round(CV_RESULTS[SELECTED_NAME]["oof_score"], 4))
print("Validated Codabench JSON:", PREDICTION_PATH)
print("Validated submission ZIP:", ZIP_PATH)
print("PASS: flat document-ID schema, exact IDs and sentence lengths, official labels, single-file ZIP, CRC valid.")

## Outputs and interpretation

The notebook saves its checkpoints, out-of-fold metrics and predictions, final training history, per-domain diagnostics, flat prediction.json, and submission.zip under /kaggle/working/phase3_rr_v1_sequence_cv. Upload only submission.zip to Codabench.

The current run shows why test-time certainty is impossible: test labels are hidden. The prior 0.3671 score is a single 10-document development estimate; this notebook uses 80 train documents for document-level cross-validation and then includes all 90 labeled documents in the final refit. Compare the out-of-fold result as the more robust estimate, not as a promise of test score.

The V1 checkpoint is loaded strictly and frozen. The notebook never runs Phase 1 training. It trains only new Phase 3 sequence-classification heads.

Research references used in the experiment:
1. Malik et al. (2022), “Semantic Segmentation of Legal Documents via Rhetorical Roles,” ACL Anthology, https://aclanthology.org/2022.nllp-1.13/. Defines the 13-role document segmentation setting and studies document-level label shift.
2. Jain et al. (2023), “NITS_Legal at SemEval-2023 Task 6,” ACL Anthology, https://aclanthology.org/2023.semeval-1.103/. Reports that sentence-sequence context is important and uses a BiLSTM over contextualized legal sentence embeddings.
3. Tandon and Chatterjee (2023), “LRL_NC at SemEval-2023 Task 6: Sequential Sentence Classification for Legal Documents Using Topic Modeling Features,” ACL Anthology, https://aclanthology.org/2023.semeval-1.20/. Uses contextual sentence representations followed by a CRF for sequence labeling.
4. Jin and Wang (2023), “TeamShakespeare at SemEval-2023 Task 6,” ACL Anthology, https://aclanthology.org/2023.semeval-1.72/.